# 05a. Modelo CF Basado en Ítems

Implementación base de filtrado colaborativo item-based usando vecinos cercanos sobre una matriz dispersa de interacciones.


In [1]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings('ignore')

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 5)

import numpy as np

from src.utils.recsys_utils import (
    build_item_knn_model,
    ensure_dir,
    make_binary_ground_truth,
    precision_recall_ndcg_at_k,
)

processed_dir = ROOT / 'data' / 'processed'
reports_dir = ensure_dir(ROOT / 'reports')
tables_dir = ensure_dir(reports_dir / 'tables')
items_dir = ensure_dir(ROOT / 'models')

In [2]:
train = pd.read_parquet(processed_dir / 'interactions_train.parquet')
valid = pd.read_parquet(processed_dir / 'interactions_valid.parquet')
test = pd.read_parquet(processed_dir / 'interactions_test.parquet')
item_index = pd.read_parquet(processed_dir / 'item_index.parquet')

## Reducir a un subconjunto denso para prototipado reproducible

El dataset completo es grande, así que se entrena sobre usuarios e ítems con mayor actividad para mantener el notebook razonable.


In [3]:
user_activity = train.groupby('userId').size().sort_values(ascending=False)
item_activity = train.groupby('movieId').size().sort_values(ascending=False)

# Prototipo liviano para notebook: se priorizan usuarios e items mas densos
active_users = user_activity.loc[lambda s: s >= 80].head(4000).index
active_items = item_activity.loc[lambda s: s >= 250].head(3000).index
train_dense = train[train['userId'].isin(active_users) & train['movieId'].isin(active_items)].copy()
valid_dense = valid[valid['userId'].isin(active_users) & valid['movieId'].isin(active_items)].copy()
test_dense = test[test['userId'].isin(active_users) & test['movieId'].isin(active_items)].copy()

for frame in [train_dense, valid_dense, test_dense]:
    for col in ['user_idx', 'item_idx']:
        if col in frame.columns:
            frame.drop(columns=col, inplace=True)

reuser = pd.DataFrame({'userId': sorted(train_dense['userId'].unique())})
reuser['user_idx'] = np.arange(len(reuser))
reitem = pd.DataFrame({'movieId': sorted(train_dense['movieId'].unique())})
reitem['item_idx'] = np.arange(len(reitem))

train_dense = train_dense.merge(reuser, on='userId', how='inner').merge(reitem, on='movieId', how='inner')
valid_dense = valid_dense.merge(reuser, on='userId', how='inner').merge(reitem, on='movieId', how='inner')
test_dense = test_dense.merge(reuser, on='userId', how='inner').merge(reitem, on='movieId', how='inner')

print(train_dense.shape, valid_dense.shape, test_dense.shape)
print('users:', train_dense['user_idx'].nunique(), 'items:', train_dense['item_idx'].nunique())
print(train_dense[['userId', 'movieId', 'user_idx', 'item_idx']].head())


(3641995, 6) (52151, 6) (44215, 6)
users: 4000 items: 3000
   userId  movieId  user_idx  item_idx
0   84603      296      1708       176
1   84603      608      1708       343
2   84603      778      1708       391
3   84603       29      1708        27
4   84603       81      1708        59


## Entrenar modelo y generar recomendaciones

La recomendación usa películas bien valoradas por cada usuario como semillas y agrega score por similitud de vecinos.


Nota de tuning: esta version queda recortada para que el notebook responda mas rapido.

Si se quiere priorizar metrica por encima de tiempo de ejecucion, se puede volver al enfoque mas pesado aumentando: `n_neighbors` del modelo, cantidad de vecinos consultados por item, cantidad de usuarios/items densos incluidos, y numero de items positivos usados por usuario como semillas.

El approach original que demora mas consistia en usar umbrales mas laxos para incluir mas usuarios e items, `n_neighbors=40`, consulta de `21` vecinos por item y recorrer todos los items positivos del usuario en vez de truncar a `20`.


In [11]:
model, item_matrix = build_item_knn_model(
    train_dense,
    n_users=train_dense['user_idx'].nunique(),
    n_items=train_dense['item_idx'].nunique(),
    n_neighbors=40,
)

positive_train = train_dense[train_dense['rating'] >= 4.0].copy()
seen_items = positive_train.groupby('user_idx')['item_idx'].apply(set).to_dict()
eval_users = set(test_dense['user_idx'].unique())
positive_train = positive_train[positive_train['user_idx'].isin(eval_users)].copy()

neighbor_cache = {}
for item_idx in train_dense['item_idx'].unique():
    distances, indices = model.kneighbors(item_matrix[item_idx], n_neighbors=21)
    neighbor_cache[int(item_idx)] = [
        (int(idx), float(1 - dist))
        for idx, dist in zip(indices[0], distances[0])
        if int(idx) != int(item_idx)
    ]

recommendation_rows = []
for user_idx, user_items in positive_train.groupby('user_idx')['item_idx']:
    scores = {}
    for item_idx in user_items.tolist()[:20]:
        for neighbor_idx, similarity in neighbor_cache[int(item_idx)]:
            if neighbor_idx in seen_items.get(int(user_idx), set()):
                continue
            scores[neighbor_idx] = scores.get(neighbor_idx, 0.0) + similarity
    top_items = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:10]
    for rank, (item_idx, score) in enumerate(top_items, start=1):
        recommendation_rows.append({'user_idx': int(user_idx), 'item_idx': int(item_idx), 'score': float(score), 'rank': rank})

recommendations_cf = pd.DataFrame(recommendation_rows)
display(recommendations_cf.head())


[22:16:46] INFO - Entrenando modelo item-kNN | filas_interacciones=3641995 | users=4000 | items=3000 | neighbors=40
[22:16:47] INFO - Modelo item-kNN listo


,user_idx,item_idx,score,rank
0,0,1291,8.195380,1
1,0,573,1.018495,2
2,0,576,1.002980,3
3,0,1958,0.814351,4
4,0,1401,0.778839,5


## Evaluación sobre test

Medimos ranking top-10 con relevancia binaria (`rating >= 4`).


In [13]:
ground_truth = make_binary_ground_truth(test_dense, threshold=4.0)
metrics = precision_recall_ndcg_at_k(recommendations_cf, ground_truth, k=10)
metrics_cf = pd.DataFrame([{'model': 'cf_item_knn', **metrics}])
display(metrics_cf)

recommendations_cf.to_parquet(items_dir / 'cf_recommendations.parquet', index=False)
metrics_cf.to_csv(tables_dir / 'cf_metrics.csv', index=False)


,model,precision_at_k,recall_at_k,ndcg_at_k,evaluated_users
0,cf_item_knn,0.044608,0.026028,0.053854,881
